In [39]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
import cloudpickle

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline

# 01. Importación
---

## 1.1. Importación de datos

In [40]:
# Definir rutas
data_path = Path('../02_datos/01_Originales')
nombre_fichero = 'Leads.csv'

# Importar Leads.csv
leads_file = data_path / nombre_fichero

df = pd.read_csv(leads_file, sep=';', encoding='utf-8', index_col='id')

print(f'✅ Importación completada')
print(f'Shape: {df.shape}')

✅ Importación completada
Shape: (9093, 20)


## 1.2. Seleccionar variables finales

In [41]:
ruta_proyecto = "../02_datos/03_Entrenamiento/"
nombres_variables_finales = ruta_proyecto + '07_df_final.pkl'

pd.read_pickle(nombres_variables_finales).sort_index().columns.to_list()

['tiempo_en_site_total_mms',
 'score_actividad_mms',
 'ult_actividad_SMS Sent',
 'paginas_vistas_visita_mms',
 'visitas_total_mms',
 'score_perfil_mms',
 'ocupacion_Working Professional',
 'ambito_Select',
 'ult_actividad_Chat Conversation',
 'ocupacion_Unemployed',
 'ult_actividad_Page Visited on Website',
 'ult_actividad_Converted to Lead',
 'descarga_lm_No',
 'compra']

In [42]:
variables_finales = ['ambito',
                     'descarga_lm',
                     'ocupacion',
                     'paginas_vistas_visita',
                     'score_actividad',
                     'score_perfil', 
                     'tiempo_en_site_total', 
                     'ult_actividad',
                     'visitas_total']

# 02. Calidad de Datos
---

![alt text](<Captura de pantalla 2026-06-15 a las 15.33.10.png>)

### Duplicados

In [43]:
df.drop_duplicates(inplace=True)

### Por EDA

In [44]:
df = df.loc[(df.no_llamar != 'OTROS') & (df.no_enviar_email != 'Yes') & (df.ult_actividad != 'Email Bounced')]

# 03. Divir en X e Y
---

In [45]:
x = df[variables_finales].copy()

In [46]:
target = 'compra'
y = df[target].copy()

# 04. Pipeline
---

## 4.1. Calidad de Datos

In [47]:
# 2. Separamos en categóricas y numéricas
cat = df.select_dtypes(exclude = 'number').copy()
num = df.select_dtypes(include = 'number').copy()

In [48]:
x.info()

<class 'pandas.DataFrame'>
Index: 6840 entries, 660737 to 579533
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   ambito                 6231 non-null   str    
 1   descarga_lm            6840 non-null   str    
 2   ocupacion              5148 non-null   str    
 3   paginas_vistas_visita  6730 non-null   float64
 4   score_actividad        3872 non-null   float64
 5   score_perfil           3872 non-null   float64
 6   tiempo_en_site_total   6840 non-null   int64  
 7   ult_actividad          6750 non-null   str    
 8   visitas_total          6730 non-null   float64
dtypes: float64(4), int64(1), str(4)
memory usage: 534.4 KB


In [49]:
def calidad_datos(df):
    # 1. Modificamos tipos
    temp = df.astype({'visitas_total': 'Int64'})

    # 2. Imputación por moda
    var_imputar_moda = ['ocupacion', 'ambito']
    def imputar_moda(variable):
        return(variable.fillna(variable.mode()[0]))

    temp[var_imputar_moda] = temp[var_imputar_moda].apply(imputar_moda)
    
    # 3. Imputación por valor
    valor = "DESCONOCIDO"
    var_imputar_valor = ['descarga_lm', 'ult_actividad']

    temp[var_imputar_valor] = temp[var_imputar_valor].fillna(valor)

    # 4. Imputación por mediana
    var_imputar_mediana = ['paginas_vistas_visita', 'score_actividad', 'score_perfil', 'tiempo_en_site_total', 'visitas_total']

    def imputar_mediana (variable):
        if pd.api.types.is_int64_dtype:
            return(variable.fillna(int(variable.median())))
    
        else:
            return variable.fillna(variable.median())

    temp[var_imputar_mediana] = temp[var_imputar_mediana].apply(imputar_mediana)


    # 5. Atípicos y categorías raras
    def agrupar_cat_raras(variable, criterio = 0.05):
        #Calcula las frecuencias
        frecuencias = variable.value_counts(normalize=True)

        #Identifica las que están por debajo 
        raras = frecuencias[frecuencias < criterio].index

        return np.where(variable.isin(raras), 'OTROS', variable)

    variables_agrupar_categorias_raras = ['ocupacion', 'ambito', 'descarga_lm', 'ult_actividad']
    criterio_agrupar = 0.02

    for variable in variables_agrupar_categorias_raras:
        temp[variable] = agrupar_cat_raras(temp[variable], criterio=criterio_agrupar)

    # 6. Winsorizacion manual
    df['visitas_total'] = df['visitas_total'].clip(0, 50)
    df['paginas_vistas_visita'] = df['paginas_vistas_visita'].clip(0, 20)

    # Imputar nulos en score_actividad y score_perfil por ceros
    cols_imputar = ['score_actividad', 'score_perfil']

    imputado = df[cols_imputar].isnull().any(axis=1).astype(int)

    # crear variable usuario_nuevo
    df['usuario_nuevo'] = imputado
    
    return(temp)

### 4.1. Convertirmos a `Transformer`

In [50]:
hacer_calidad_datos = FunctionTransformer(calidad_datos)

# 7. Transformación

### 7.1. Instanciamos One-Hot Encoding & Min-Max Scaling

In [51]:
var_ohe = ['ambito', 'descarga_lm', 'ocupacion', 'ult_actividad']
var_mms = ['paginas_vistas_visita', 'score_actividad', 'score_perfil', 'visitas_total', 'tiempo_en_site_total']

# Instanciamos OHE & M-M Scaling
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler

ohe = OneHotEncoder(sparse_output = False, handle_unknown = 'ignore')
mms = MinMaxScaler()

### 7.2. Aplicamos OHE y M-M Scaling

In [52]:
ct = make_column_transformer(
    (ohe,var_ohe),
    (mms, var_mms),
    remainder='drop'
)

### 7.3. Crear pipeline del procesamiento

In [53]:
pipe_prepro = make_pipeline(hacer_calidad_datos, 
                            ct)

# 7. Modelización
---

Recordamos que el mejor modelo al aplicar este código `modelo.best_params_` tuvo esta salida:

{'algoritmo': LogisticRegression(),
 'algoritmo__C': 1,
 'algoritmo__n_jobs': -1,
 'algoritmo__penalty': 'l1',
 'algoritmo__solver': 'saga'}

## 7.1. Construimos el Pipeline del modelo

### 7.1.1. Instanciamos el modelo final

In [54]:
from sklearn.linear_model import LogisticRegression

# Instanciamos
modelo = LogisticRegression(
 C = 1,
 n_jobs =  -1,
 penalty = 'l1',
 solver = 'saga',
)

### 7.1.2. Creamos el pipeline de entrenamiento

In [55]:
# Crear el pipeline de entrenamiento
pipe_entrenamiento = make_pipeline(pipe_prepro, modelo)

In [ ]:
# Guardamos el pipeline final de entrenamiento
ruta_entrenamiento = '../05_modelos/'
nombre_pipe_entrenamiento = 'pipe_entrenamiento.pkl'

ruta_pipe_entrenmiento = ruta_entrenamiento + nombre_pipe_entrenamiento

with open(ruta_pipe_entrenmiento, mode='wb') as file:
    cloudpickle.dump(pipe_entrenamiento, file)

### 7.1.3. Creamos el pipeline de ejecución

In [57]:
x.info()

<class 'pandas.DataFrame'>
Index: 6840 entries, 660737 to 579533
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   ambito                 6231 non-null   str    
 1   descarga_lm            6840 non-null   str    
 2   ocupacion              5148 non-null   str    
 3   paginas_vistas_visita  6730 non-null   float64
 4   score_actividad        3872 non-null   float64
 5   score_perfil           3872 non-null   float64
 6   tiempo_en_site_total   6840 non-null   int64  
 7   ult_actividad          6750 non-null   str    
 8   visitas_total          6730 non-null   float64
dtypes: float64(4), int64(1), str(4)
memory usage: 534.4 KB


In [58]:
pipe_ejecucion = pipe_entrenamiento.fit(x, y)

/Users/davidsantossalvador/miniconda3/envs/01_LeadScoring2/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/davidsantossalvador/miniconda3/envs/01_LeadScoring2/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/Users/davidsantossalvador/miniconda3/envs/01_LeadScoring2/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarn

## 7.2. Guardar el Pipeline

In [60]:
nombre_pipe_ejecucion = 'pipe_ejecucion.pkl'

In [61]:
# Guardamos el pipeline final de entrenamiento
ruta_ejecucion = '../05_modelos/'
nombre_pipe_ejecicion = 'pipe_entrenamiento.pkl'

ruta_pipe_entrenmiento = ruta_ejecucion + nombre_pipe_ejecucion

with open(ruta_pipe_entrenmiento, mode='wb') as file:
    cloudpickle.dump(pipe_ejecucion, file)